# From Data to AI: NovaBank Controls Assistant
**ReadyScale · Hands-on notebook for Video 1** · Trainer: Harshika Pareek

In this notebook you'll build, step by step, an AI assistant for **NovaBank**, a fictional bank. Its Global Finance Controls team reviews flagged card transactions and wants an assistant that explains *why* a transaction was flagged and *which policy* applies.

| Step | What you'll do | Concept from the video |
|---|---|---|
| 1 | Load messy transactions, run quality checks, clean them | The five data quality checks |
| 2 | Train a fraud model and read its results properly | The accuracy trap, precision vs recall |
| 3 | Build RAG over NovaBank's policy documents | How RAG works, bad documents give bad answers |
| 4 | Run an eval: fail → fix the data → pass | Eval is control testing for AI |
| 5 | Launch a chatbot your analysts can use | Putting it all together |

### Before you start
1. **Make your own copy:** *File → Save a copy in Drive*. Then you can edit and run everything.
2. **Run the cells in order**, top to bottom (Shift + Enter runs one cell).
   *Running locally (VS Code/Jupyter) instead?* Install the libraries once with `pip install -r requirements.txt` from the repo, and set `GEMINI_API_KEY` as an environment variable.
3. **Optional but recommended: a free Gemini API key.** Without one, the notebook still runs end to end, but steps 3–5 return quoted policy text instead of AI-written answers.
   - Get a key at [Google AI Studio](https://aistudio.google.com/apikey).
   - In Colab, click the 🔑 **Secrets** icon in the left sidebar → **Add new secret** → name it `GEMINI_API_KEY` → paste the key → switch on **Notebook access**.
   - Never paste your key directly into a code cell or share it.

> All data in this notebook is synthetic. NovaBank, its customers and its policies are fictional.

## 0 · Setup

In [ ]:
# Install the few libraries Colab doesn't already have (takes ~1 minute the first time)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "gradio", "google-genai", "sentence-transformers"], check=True)

import re, os, time, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.ERROR)   # hide advisory messages from the Gemini library
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# ReadyScale brand colours for charts
ORANGE, YELLOW, GREY = "#FF6B00", "#FFD700", "#8A8A8A"
SEED = 42
AS_OF = pd.Timestamp("2026-07-01")   # "today" for this exercise
print("Setup complete. Running in Colab:", IN_COLAB)

In [ ]:
# Connect to Gemini if you've added a GEMINI_API_KEY secret (optional)
# Google retires older models over time. We try the recommended model first, then any other Flash model your key can use.
# To force a specific model, set GEMINI_MODEL (e.g. "gemini-3.8-flash"); leave it as None for automatic.
GEMINI_MODEL = "gemini-3.1-flash-lite"
PREFERRED_MODELS = ["gemini-3.8-flash"]

def call_gemini(client, model, prompt):
    """Send one prompt and return the text. Uses the Interactions API, or generate_content on older SDKs."""
    if hasattr(client, "interactions"):
        try:
            return client.interactions.create(model=model, input=prompt).output_text.strip()
        except Exception as e:
            if any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED")):
                raise
    return client.models.generate_content(model=model, contents=prompt).text.strip()

def _version(name):
    m = re.search(r"(\d+(?:\.\d+)?)", name)
    return float(m.group(1)) if m else 0.0

def candidate_models(client):
    """Flash models newest first, then Flash-Lite models (smaller, but with higher free-tier limits)."""
    flash, lite = [], []
    try:
        for m in client.models.list():
            name = m.name.split("/")[-1]
            if "flash" not in name or any(x in name for x in ("image", "tts", "live", "audio", "embedding", "exp")):
                continue
            (lite if "lite" in name else flash).append(name)
    except Exception:
        pass
    order = lambda n: ("preview" in n, -_version(n))
    return list(dict.fromkeys(PREFERRED_MODELS + sorted(flash, key=order) + sorted(lite, key=order)))

llm = None
BACKUP_MODELS = []   # used automatically if the main model runs out of free quota
try:
    key = None
    if IN_COLAB:
        from google.colab import userdata
        try:
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None
    key = key or os.environ.get("GEMINI_API_KEY")
    if key:
        from google import genai
        client = genai.Client(api_key=key)
        errors = []
        candidates = [GEMINI_MODEL] if GEMINI_MODEL else candidate_models(client)
        for i, name in enumerate(candidates[:6]):
            try:
                call_gemini(client, name, "Reply with OK")
                GEMINI_MODEL, llm = name, client
                BACKUP_MODELS = candidates[i + 1:]
                break
            except Exception as e:
                errors.append(f"{name}: {str(e)[:250]}")
        if llm:
            print(f"Gemini connected ({GEMINI_MODEL}). Answers will be AI-generated.")
        else:
            print("Could not connect to Gemini. Errors per model tried:")
            print("\n".join("  - " + e for e in errors) or "  - no Flash models found for this key")
except Exception as e:
    llm = None
    print("Could not connect to Gemini:", str(e)[:300])
if llm is None:
    print("Running WITHOUT an LLM: answers will quote the retrieved policy text. Everything else works the same.")

---
## Step 1 · Data first
**Remember the five data quality checks?** Complete, accurate, consistent, timely, unique. Let's see what happens when real-looking data breaks them.

First, we generate six months of NovaBank card transactions. Real exports are messy, so this one has problems built in, just like the systems you'll meet at work.

In [ ]:
def generate_transactions(seed=SEED):
    rng = np.random.default_rng(seed)
    n_accounts = 400
    acct_ids = [f"A{i:03d}" for i in range(1, n_accounts + 1)]
    typical = dict(zip(acct_ids, rng.lognormal(mean=3.6, sigma=0.5, size=n_accounts)))  # ~€35 typical spend

    merchants = {
        "groceries":   ["FreshMart", "Daily Grocer", "GreenBasket"],
        "restaurants": ["Harbour Bistro", "Noodle Bar", "Café Central"],
        "transport":   ["CityRail", "QuickCab", "FuelStop"],
        "retail":      ["StyleHouse", "HomeGoods Co", "BookNook"],
        "electronics": ["GlobalTech Electronics", "GadgetWorld", "PixelPoint"],
        "travel":      ["SkyHigh Airlines", "StayWell Hotels", "TripDesk"],
    }
    cats = list(merchants)
    overseas = ["GB", "FR", "ES", "DE", "US", "AE"]
    start = pd.Timestamp("2026-01-01")
    hour_p = np.array([1,1,1,1,1,2,4,6,7,7,7,8,8,7,7,7,7,7,7,6,5,4,3,2], float); hour_p /= hour_p.sum()

    rows = []
    # Normal transactions
    for _ in range(4900):
        a = rng.choice(acct_ids)
        cat = rng.choice(cats, p=[.30, .20, .18, .17, .08, .07])
        ts = start + pd.Timedelta(days=int(rng.integers(0, 181)), hours=int(rng.choice(24, p=hour_p)),
                                  minutes=int(rng.integers(0, 60)))
        rows.append(dict(timestamp=ts, account_id=a, merchant=rng.choice(merchants[cat]), category=cat,
                         amount_eur=round(typical[a] * rng.lognormal(0, 0.6) * (4 if cat == "travel" else 1), 2),
                         country="IE" if rng.random() < 0.9 else rng.choice(overseas),
                         channel="card_not_present" if rng.random() < 0.35 else "card_present", is_fraud=0))
    # Fraud events: bursts of 1–3 transactions within minutes
    for _ in range(40):
        a = rng.choice(acct_ids)
        hour = int(rng.integers(0, 6)) if rng.random() < 0.6 else int(rng.choice(24, p=hour_p))
        base = start + pd.Timedelta(days=int(rng.integers(0, 181)), hours=hour, minutes=int(rng.integers(0, 60)))
        country = rng.choice(overseas) if rng.random() < 0.8 else "IE"
        for k in range(int(rng.integers(1, 4))):
            cat = rng.choice(["electronics", "retail", "travel"])
            rows.append(dict(timestamp=base + pd.Timedelta(minutes=int(k * rng.integers(3, 15))), account_id=a,
                             merchant=rng.choice(merchants[cat]), category=cat,
                             amount_eur=round(typical[a] * rng.uniform(4, 12), 2), country=country,
                             channel="card_not_present" if rng.random() < 0.8 else "card_present", is_fraud=1))

    df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    df.insert(0, "transaction_id", [f"T{1000 + i}" for i in range(len(df))])
    # The transaction we'll investigate in the chatbot
    df.loc[df.transaction_id == "T1023", ["timestamp", "merchant", "category", "amount_eur", "country", "channel", "is_fraud"]] = \
        [pd.Timestamp("2026-01-03 03:14"), "GlobalTech Electronics", "electronics", 6200.00, "US", "card_not_present", 1]

    # --- Real-world mess -------------------------------------------------
    raw = df.copy()
    raw["currency"] = "EUR"
    raw["amount"] = raw["amount_eur"]
    protect = raw.transaction_id == "T1023"
    fx = {"USD": 0.92, "GBP": 1.17}                       # EUR per unit
    idx = raw[~protect].sample(frac=0.08, random_state=1).index          # mixed currencies
    cur = rng.choice(["USD", "GBP"], size=len(idx))
    raw.loc[idx, "currency"] = cur
    raw.loc[idx, "amount"] = (raw.loc[idx, "amount_eur"] / pd.Series(cur, index=idx).map(fx)).round(2)
    raw.loc[raw[~protect].sample(frac=0.02, random_state=2).index, "merchant"] = None       # missing values
    neg = raw[~protect].sample(frac=0.005, random_state=3).index                             # sign errors
    raw.loc[neg, "amount"] = -raw.loc[neg, "amount"]
    fut = raw[~protect].sample(frac=0.004, random_state=4).index                             # bad dates
    raw.loc[fut, "timestamp"] = raw.loc[fut, "timestamp"] + pd.DateOffset(years=1)
    dups = raw.sample(frac=0.03, random_state=5)                                             # double-loaded rows
    raw = pd.concat([raw, dups]).sort_values("transaction_id", kind="stable").reset_index(drop=True)
    return raw[["transaction_id", "timestamp", "account_id", "merchant", "category",
                "amount", "currency", "country", "channel", "is_fraud"]]

generate_transactions().to_csv("novabank_transactions_raw.csv", index=False)

# In real life, this is where you'd load an export from a source system
raw = pd.read_csv("novabank_transactions_raw.csv", parse_dates=["timestamp"])
print(f"Loaded {len(raw):,} rows")
raw.head(8)

### Run the five quality checks
Each check is a simple rule. Together, they decide whether we can trust anything built on top of this data.

In [ ]:
FX_TO_EUR = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17}

def run_quality_checks(df):
    checks = [
        ("Complete",   "Merchant name is missing",        df["merchant"].isna()),
        ("Accurate",   "Amount is zero or negative",      df["amount"] <= 0),
        ("Consistent", "Amount not in EUR",               df["currency"] != "EUR"),
        ("Timely",     "Timestamp is in the future",      df["timestamp"] > AS_OF),
        ("Unique",     "Duplicate transaction ID",        df.duplicated("transaction_id", keep="first")),
    ]
    report = pd.DataFrame([(dim, rule, int(mask.sum()), f"{mask.mean():.1%}") for dim, rule, mask in checks],
                          columns=["Check", "Rule", "Rows failing", "% of rows"])
    report["Status"] = np.where(report["Rows failing"] == 0, "✅ PASS", "❌ FAIL")
    return report

run_quality_checks(raw)

**Why does this matter?** Look at what a naive total-spend report would say, compared with the truth:

In [ ]:
naive_total = raw["amount"].sum()                       # what a quick report would show
dups_mask = raw.duplicated("transaction_id", keep="first")
dup_overstatement = (raw.loc[dups_mask, "amount"] * raw.loc[dups_mask, "currency"].map(FX_TO_EUR)).sum()
print(f"Naive total 'spend' (mixed currencies, duplicates, sign errors): €{naive_total:,.0f}")
print(f"Duplicates alone overstate spend by:                         €{dup_overstatement:,.0f}")

### Clean the data, the controls way
In finance we don't silently delete or "fix" suspicious records. We **quarantine** them for review, so there's an audit trail of what was excluded and why.

In [ ]:
def clean_transactions(df):
    df = df.drop_duplicates("transaction_id", keep="first").copy()
    reasons = np.select([df["amount"] <= 0, df["timestamp"] > AS_OF],
                        ["Non-positive amount", "Future-dated"], default="")
    quarantine = df[reasons != ""].assign(quarantine_reason=reasons[reasons != ""])
    clean = df[reasons == ""].copy()
    clean["amount_eur"] = (clean["amount"] * clean["currency"].map(FX_TO_EUR)).round(2)
    clean["original_currency"] = clean["currency"]
    clean["currency"], clean["amount"] = "EUR", clean["amount_eur"]
    clean["merchant"] = clean["merchant"].fillna("UNKNOWN")
    return clean.reset_index(drop=True), quarantine

clean, quarantine = clean_transactions(raw)
print(f"Clean rows: {len(clean):,}   Quarantined for review: {len(quarantine)}")
display(quarantine[["transaction_id", "timestamp", "amount", "currency", "quarantine_reason"]].head())
run_quality_checks(clean)

> **Your turn:** add a sixth check to `run_quality_checks`, for example *"Country code is not two letters"* (`~df["country"].str.fullmatch("[A-Z]{2}")`). Re-run both reports.

---
## Step 2 · From data to ML
**Remember: features are where data engineering becomes AI.** We turn each transaction into signals a model can learn from.

In [ ]:
def build_features(df):
    df = df.sort_values(["account_id", "timestamp"]).copy()
    df["amount_vs_usual"] = df["amount_eur"] / df.groupby("account_id")["amount_eur"].transform("median")
    df["is_overseas"] = (df["country"] != "IE").astype(int)
    df["is_night"] = (df["timestamp"].dt.hour < 6).astype(int)
    df["is_card_not_present"] = (df["channel"] == "card_not_present").astype(int)
    def count_last_hour(g):
        t = g["timestamp"].values.astype("int64")
        return pd.Series(np.arange(len(t)) - np.searchsorted(t, t - 3_600 * 10**9) + 1, index=g.index)
    df["txns_last_hour"] = df.groupby("account_id", group_keys=False).apply(count_last_hour)
    return df.sort_values("transaction_id").reset_index(drop=True)

FEATURES = ["amount_eur", "amount_vs_usual", "is_overseas", "is_night", "is_card_not_present", "txns_last_hour"]
data = build_features(clean)
print(f"Fraud rate: {data['is_fraud'].mean():.1%} of transactions")
data[["transaction_id"] + FEATURES + ["is_fraud"]].head()

### The accuracy trap
First, a "model" that always says *not fraud*. Then a real one.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

X, y = data[FEATURES], data["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)

lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=SEED).fit(X_train, y_train)

THRESHOLD = 0.5   # flag a transaction when its risk score is at or above this

def evaluate(name, scores):
    pred = (scores >= THRESHOLD).astype(int)
    return {"Model": name, "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "Fraud caught": f"{((pred == 1) & (y_test == 1)).sum()} of {int(y_test.sum())}"}

results = pd.DataFrame([evaluate("Always 'not fraud'", lazy.predict_proba(X_test)[:, 1]),
                        evaluate("Random forest", model.predict_proba(X_test)[:, 1])])
results.style.format({"Accuracy": "{:.1%}", "Precision": "{:.1%}", "Recall": "{:.1%}"}).hide(axis="index")

Both look "accurate". Only one catches fraud. Let's look closer at the real model:

In [ ]:
pred = (model.predict_proba(X_test)[:, 1] >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = np.array([[tn, fp], [fn, tp]])
axes[0].imshow(cm, cmap="Oranges")
labels = [["Correctly cleared", "False alarm\n(customer blocked)"], ["Missed fraud\n(money lost)", "Fraud caught"]]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm[i, j]}\n{labels[i][j]}", ha="center", va="center", fontsize=10,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
axes[0].set_xticks([0, 1], ["Predicted: OK", "Predicted: fraud"]); axes[0].set_yticks([0, 1], ["Actual: OK", "Actual: fraud"])
axes[0].set_title("Confusion matrix")

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
axes[1].barh(imp.index, imp.values, color=ORANGE); axes[1].set_title("What the model relies on")
axes[1].spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
print(f"Precision {tp / (tp + fp):.0%}: of flagged transactions, how many were really fraud")
print(f"Recall    {tp / (tp + fn):.0%}: of all real fraud, how much we caught")

> **Your turn:** change `THRESHOLD` to `0.3`, then `0.8`, and re-run the last two cells. Watch precision and recall trade off. Which would you choose if a missed fraud costs €5,000 and a false alarm costs €20 of analyst time?

In [ ]:
# Score every transaction so the assistant can explain them later
data["risk_score"] = model.predict_proba(data[FEATURES])[:, 1].round(2)
data.loc[data.transaction_id == "T1023", ["transaction_id", "timestamp", "merchant", "amount_eur", "country",
                                          "channel", "txns_last_hour", "risk_score"]]

---
## Step 3 · From data to GenAI: RAG
The model gives a score, but analysts also need to know **which policy applies**. That knowledge lives in documents, so we'll use **RAG**: retrieve the relevant policy passages, then let the LLM answer from them.

Here are NovaBank's policy documents. Notice the overseas verification policy has **two versions**.

In [ ]:
POLICIES = [
    {"policy_id": "CTM-2024", "family": "CTM", "title": "Card Transaction Monitoring Policy", "version": 3,
     "effective": "2024-09-01", "status": "active", "sections": {
        "2.1": "Every card transaction is scored by the fraud model. Transactions with a risk score of 0.80 or higher are flagged for analyst review, which must be completed within 24 hours.",
        "2.2": "Three or more card transactions on the same account within one hour trigger a velocity alert, and the account is reviewed by an analyst.",
        "2.3": "Transactions between 00:00 and 05:59 at a merchant the customer has not used before receive additional review."}},
    {"policy_id": "HVP-2025", "family": "HVP", "title": "High-Value Payment Approval Policy", "version": 2,
     "effective": "2025-02-01", "status": "active", "sections": {
        "4.1": "Payments of €25,000 or more require approval by two authorised approvers (four-eyes principle).",
        "4.2": "Any payment over €7,500 to a new payee requires call-back verification with the payee, using contact details already on file, before the payment is released."}},
    {"policy_id": "SOD-2024", "family": "SOD", "title": "Segregation of Duties Policy", "version": 1,
     "effective": "2024-11-01", "status": "active", "sections": {
        "5.1": "The person who creates or edits a vendor record cannot approve payments to that vendor.",
        "5.2": "An analyst who raises a fraud escalation cannot close their own escalation; a second analyst must review it.",
        "5.3": "Access rights for payment and vendor systems are reviewed quarterly by the system owner."}},
    {"policy_id": "OTV-2022", "family": "OTV", "title": "Overseas Transaction Verification Policy", "version": 1,
     "effective": "2022-03-01", "status": "superseded", "sections": {
        "3.2": "Overseas card transactions (made abroad, outside Ireland) above €10,000 require verification with the customer before approval."}},
    {"policy_id": "OTV-2024", "family": "OTV", "title": "Overseas Transaction Verification Policy", "version": 2,
     "effective": "2024-06-01", "status": "active", "sections": {
        "3.2": "Overseas card transactions (made abroad, outside Ireland) above €5,000 require verification with the customer, by in-app confirmation or phone, before approval.",
        "3.3": "Overseas card-not-present transactions above €2,000 require a 3-D Secure step-up check."}},
]

# The knowledge base as it was first loaded, from an old shared-drive export
OLD_EXPORT = ["CTM-2024", "HVP-2025", "SOD-2024", "OTV-2022"]
pd.DataFrame(POLICIES)[["policy_id", "title", "version", "effective", "status"]]

### Chunk and embed
Each policy section becomes one chunk. Then each chunk becomes an **embedding**: a list of numbers that captures its meaning.

In [ ]:
def chunk_policies(policy_ids):
    return [{"policy_id": p["policy_id"], "section": s, "title": p["title"], "text": t}
            for p in POLICIES if p["policy_id"] in policy_ids for s, t in p["sections"].items()]

try:
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer("all-MiniLM-L6-v2")
    def embed(texts):
        return _st.encode(texts, normalize_embeddings=True)
    EMBEDDER = "sentence-transformers (all-MiniLM-L6-v2)"
except Exception:
    # Fallback if the embedding model can't be downloaded: word-based TF-IDF vectors
    from sklearn.feature_extraction.text import TfidfVectorizer
    _all_text = [t for p in POLICIES for t in p["sections"].values()]
    _tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2)).fit(_all_text)
    def embed(texts):
        v = _tfidf.transform(texts).toarray()
        return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
    EMBEDDER = "TF-IDF fallback"

class PolicyIndex:
    def __init__(self, policy_ids):
        self.chunks = chunk_policies(policy_ids)
        self.vectors = embed([f"{c['title']}. {c['text']}" for c in self.chunks])
    def search(self, question, k=2):
        scores = self.vectors @ embed([question])[0]
        return [dict(self.chunks[i], score=float(scores[i])) for i in np.argsort(-scores)[:k]]

index = PolicyIndex(OLD_EXPORT)
print(f"Embedder: {EMBEDDER} · {len(index.chunks)} chunks · each embedding has {index.vectors.shape[1]} numbers")
pd.DataFrame(index.search("Does a €6,200 overseas card payment need verification?"))[["policy_id", "section", "score", "text"]]

### Retrieve, then generate

In [ ]:
def quote_answer(retrieved, facts=""):
    """Answer without an LLM: show the facts and quote the retrieved policy passages."""
    if facts:
        return facts + "\n\nRelevant policies:\n" + "\n".join(f"- [{c['policy_id']} §{c['section']}] {c['text']}" for c in retrieved)
    top = retrieved[0]
    return f"Relevant policy [{top['policy_id']} §{top['section']}]: {top['text']}"

_last_call = [0.0]
def _pace(min_gap=6.5):
    """Free tier allows ~10 requests a minute, so keep at least 6.5 seconds between calls."""
    wait = _last_call[0] + min_gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

def ask_gemini(prompt):
    """Call Gemini politely: pace requests, retry when busy, switch model if one runs out of quota."""
    global GEMINI_MODEL
    failures = 0
    while True:
        _pace()
        try:
            return call_gemini(llm, GEMINI_MODEL, prompt)
        except Exception as e:
            msg = str(e)
            if not any(k in msg for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL")):
                raise
            failures += 1
            daily = "per day" in msg.lower() or "perday" in msg.lower()
            if (daily or "503" in msg or failures >= 2) and BACKUP_MODELS:
                print(f"  ({GEMINI_MODEL} limit reached, switching to {BACKUP_MODELS[0]})")
                GEMINI_MODEL = BACKUP_MODELS.pop(0)
                failures = 0
            elif failures >= 3:
                return None
            else:
                print(f"  (Gemini busy: {msg[:90]}... waiting 20s)")
                time.sleep(20)

GEMINI_OFF = False   # set to True if every model has run out of free quota

def generate_answer(question, retrieved, facts=""):
    global GEMINI_OFF
    if llm is None or GEMINI_OFF:
        return quote_answer(retrieved, facts)
    context = "\n".join(f"[{c['policy_id']} §{c['section']}] {c['text']}" for c in retrieved)
    prompt = f"""You are NovaBank's controls assistant for fraud analysts.
Answer the question using ONLY the policy extracts and transaction facts below.
Cite every policy you use as [POLICY_ID §section]. If the extracts don't contain the answer, say you don't know.
Keep it to 2–4 sentences.

Policy extracts:
{context}

Transaction facts:
{facts or "None"}

Question: {question}"""
    answer = ask_gemini(prompt)
    if answer is None:
        GEMINI_OFF = True
        print("  (Gemini's free quota is used up for now: quoting policy text for the rest of this run."
              " Wait a while, then re-run cell 2 to reconnect.)")
        return quote_answer(retrieved, facts)
    return answer

def ask(question, idx, facts=""):
    return generate_answer(question, idx.search(question), facts)

print(ask("Does a €6,200 overseas card payment need verification?", index))

**Is that right?** Check the policy table above. The answer is based on the **2022** policy (€10,000 threshold), which has been replaced. The current policy says €5,000, so this €6,200 payment **does** need verification.

The model didn't fail. **The data did:** the old export never picked up the 2024 update. That's a *timeliness* problem, the same check from Step 1, applied to documents.

Let's catch it the way a controls team would: with a test.

---
## Step 4 · Eval: prove it works
**Eval is control testing for AI.** We write test questions with known correct answers, run them, score them, and only release if they pass.

*With Gemini switched on, each eval run takes about a minute: the notebook spaces out its requests to stay within the free tier's limit of ~10 per minute.*

In [ ]:
TEST_SET = [
    # question, expected policy, answer must mention (any one of the options)
    ("Does a €6,200 overseas card payment need verification?",               "OTV-2024", ["5000"]),
    ("What is the threshold for verifying overseas card transactions?",       "OTV-2024", ["5000"]),
    ("A customer spent €7,000 on their card while abroad. Does it need verification?", "OTV-2024", ["5000"]),
    ("What risk score means a transaction is flagged for analyst review?",    "CTM-2024", ["0.80", "0.8"]),
    ("How many card transactions in one hour trigger a velocity alert?",      "CTM-2024", ["three", "3"]),
    ("Within how many hours must a flagged transaction be reviewed?",         "CTM-2024", ["24"]),
    ("Which payments need approval from two approvers?",                      "HVP-2025", ["25000"]),
    ("What must happen before releasing an €8,000 payment to a new payee?",   "HVP-2025", ["call-back", "callback", "call back"]),
    ("Can the person who created a vendor approve payments to that vendor?",  "SOD-2024", ["cannot", "can not", "must not", "not allowed", "not permitted"]),
    ("How often are access rights to payment systems reviewed?",              "SOD-2024", ["quarterly", "every quarter", "every three months"]),
]
PASS_MARK = 0.9   # release gate: at least 90% on both checks

def normalise(text):
    return text.lower().replace(",", "").replace("€", "")

def run_eval(idx, label):
    rows = []
    for q, expected, must_mention in TEST_SET:
        retrieved = idx.search(q)
        answer = generate_answer(q, retrieved)
        rows.append({"Question": q, "Expected policy": expected,
                     "Retrieved": ", ".join(c["policy_id"] for c in retrieved),
                     "Right policy?": expected in [c["policy_id"] for c in retrieved],
                     "Correct answer?": any(re.search(rf"(?<![\w.]){re.escape(normalise(m))}(?![\w])", normalise(answer))
                                            for m in must_mention)})
    res = pd.DataFrame(rows)
    retrieval, correctness = res["Right policy?"].mean(), res["Correct answer?"].mean()
    passed = retrieval >= PASS_MARK and correctness >= PASS_MARK
    print(f"{label}\n  Retrieval accuracy: {retrieval:.0%}   Answer accuracy: {correctness:.0%}   Pass mark: {PASS_MARK:.0%}")
    print("  RESULT:", "✅ PASS: cleared for release" if passed else "❌ FAIL: do not release")
    return passed, res

passed, results_before = run_eval(index, "Eval #1: knowledge base from the old export")
results_before.style.map(lambda v: "color: #C62828; font-weight: bold" if v is False else "", subset=["Right policy?", "Correct answer?"])

### Fix the data, not the model
Before changing anything, run a quality check on the **documents**, just like we did on the transactions.

In [ ]:
def check_documents(policy_ids):
    docs = pd.DataFrame([p for p in POLICIES if p["policy_id"] in policy_ids])
    docs["age_months"] = ((AS_OF - pd.to_datetime(docs["effective"])).dt.days / 30.4).round()
    docs["issue"] = np.select(
        [docs["status"] != "active", docs["age_months"] > 36, docs.duplicated("family", keep=False)],
        ["Superseded version still loaded", "Not reviewed in 36+ months", "Multiple versions of same policy"], default="OK")
    return docs[["policy_id", "title", "version", "effective", "status", "age_months", "issue"]]

check_documents(OLD_EXPORT)

In [ ]:
# Fix: load the current export, then keep only the latest ACTIVE version of each policy
CURRENT_EXPORT = [p["policy_id"] for p in POLICIES]
latest = (pd.DataFrame([p for p in POLICIES if p["policy_id"] in CURRENT_EXPORT])
          .query("status == 'active'").sort_values("version").groupby("family").tail(1))
FIXED_KB = latest["policy_id"].tolist()
print("Knowledge base after fix:", FIXED_KB)
check_documents(FIXED_KB)

In [ ]:
index = PolicyIndex(FIXED_KB)
EVAL_PASSED, results_after = run_eval(index, "Eval #2: after fixing the documents")
print()
print(ask("Does a €6,200 overseas card payment need verification?", index))

Same model, same code. **Only the data changed**, and the eval now passes. This is the loop you'll use on real projects: *change → test → release*.

> **Your turn:** add two more questions to `TEST_SET` (for example about 3-D Secure, or about who closes an escalation) and re-run the eval.
>
> **Going further:** keyword checks are simple and transparent. Larger teams also use **LLM-as-judge**, where a second LLM call grades whether each answer is faithful to the retrieved policy.

---
## Step 5 · Launch the chatbot
The release gate passed, so we can put a chat window on top of everything: the clean data, the fraud model's scores, and RAG over the current policies.

In [ ]:
import gradio as gr

def transaction_facts(message):
    """If the question mentions a transaction ID, look it up and return its facts and risk factors."""
    m = re.search(r"\bT\d{4}\b", message.upper())
    if not m:
        return "", []
    row = data[data.transaction_id == m.group(0)]
    if row.empty:
        return f"Transaction {m.group(0)} was not found.", []
    r = row.iloc[0]
    facts = (f"Transaction {r.transaction_id}: €{r.amount_eur:,.2f} at {r.merchant} ({r.category}), "
             f"country {r.country}, {r.channel.replace('_', ' ')}, at {r.timestamp:%H:%M} on {r.timestamp:%d %b %Y}. "
             f"Fraud model risk score {r.risk_score:.2f}. {int(r.txns_last_hour)} transaction(s) on this account in the last hour. "
             f"Amount is {r.amount_vs_usual:.0f}x the customer's usual spend.")
    # One policy search per risk factor, so every relevant rule is retrieved
    searches = ["risk score flagged for analyst review"]
    if r.is_overseas:
        searches.append("overseas card transactions abroad require verification threshold")
    if r.is_overseas and r.is_card_not_present:
        searches.append("overseas card-not-present 3-D Secure step-up")
    if r.txns_last_hour >= 3:
        searches.append("card transactions within one hour velocity alert")
    if r.is_night:
        searches.append("transactions between 00:00 and 05:59 additional review")
    return facts, searches

def chat(message, history):
    facts, searches = transaction_facts(message)
    retrieved, seen = [], set()
    for q in [message] + searches:
        for c in index.search(q, k=1 if searches else 2):
            if (c["policy_id"], c["section"]) not in seen:
                seen.add((c["policy_id"], c["section"])); retrieved.append(c)
    answer = generate_answer(message, retrieved, facts)
    sources = ", ".join(f"{c['policy_id']} §{c['section']}" for c in retrieved)
    return f"{answer}\n\n_Sources checked: {sources}_"

if not EVAL_PASSED:
    print("⚠️ The eval has not passed. In a real project, the release gate would stop here.")

demo = gr.ChatInterface(
    fn=chat,
    title="NovaBank Controls Assistant",
    description="Ask about flagged transactions or NovaBank's controls policies. Synthetic data, for training only.",
    examples=["Why was transaction T1023 flagged?",
              "What is the rule for large overseas card payments?",
              "Can I approve an €8,000 payment to a new payee?"],
)
print(chat("Why was transaction T1023 flagged?", []))   # quick test before launching

In [ ]:
demo.launch(share=IN_COLAB)   # Colab: chat below + public link (~72h). VS Code/Jupyter: opens at http://127.0.0.1:7860

---
## Practical exercise
1. Add a new policy to `POLICIES`: **"Crypto purchases over €1,000 require review"** (give it a `policy_id`, `family`, `version`, `effective` date, `status` and one section).
2. Add it to the knowledge base and rebuild the index (`index = PolicyIndex(FIXED_KB + ["<your id>"])`).
3. Add three test questions about it to `TEST_SET`.
4. Re-run the eval until it passes.
5. Relaunch the chatbot and ask your questions.

## Key takeaways
- **Data quality decides AI quality,** for tables and for documents.
- **Pick metrics by business cost.** Accuracy hides failure on rare events.
- **RAG is a data pipeline.** Version, de-duplicate and retire old documents.
- **Never ship untested AI.** Eval is control testing for AI.

*Next course: **AI Agents for Business**, where the assistant stops just answering and starts taking actions.*